# 13 · Carry or repeat: a learned decision per tile

Every learned model that drew pixels or motion lost to the tile search, or gained a few tenths of
a dB for many times its work: two received frames do not say what happens next, and a network
unsure of it hedges. A **decision** cannot hedge. Per tile it picks one of two sharp pictures - the
search's N+1 (N moved by half the motion it found) or N itself - and a wrong pick is a whole-tile
mistake, never a blur.

The decision sees only what the client has: the search's own work on the two received frames and
the stream's bitrate (`helpers/decide.py`, `FEATURES`).

- **The control, no learning**: the search's gate raised - repeat wherever its cost at no motion is
  under a threshold, tuned on the training clips.
- **Learned**: logistic regression and a small network (12 → 16 → 16 → 1) on the 12 features,
  trained to pick the picture that scores better, each tile weighted by how much better (capped
  at 50 dB, the moving tiles' score) - on every kind of the training split, on the drawn kinds
  only and on the recorded kinds only; the step kept chosen on a held-out tenth of the training
  sources (by video for the recordings).
- **The ceiling**: the better of the two per tile, which sees N+1.

Scored on the validation split, by kind.

Needs: `00_data_preprocess.ipynb`. Nothing is exported; results cached in `checkpoints/decide/`.

In [1]:
"""Settings."""

import pickle
import time

import numpy as np
import torch

from helpers import decide, models, runs, training
from helpers.decide import FEATURES
from helpers.training import DEVICE

STILL, SHARPNESS, CAP = 0.1, 10.0, 50.0
CACHE = training.CHECKPOINT_DIR / "decide"
SEED = 0
KINDS = ("page", "desktop", "photo", "screen", "video")
DRAWN, RECORDED = ("page", "desktop", "photo"), ("screen", "video")
training.seed_everything(SEED)
manifest = training.load_manifest()
published = models.TileSearch(4, 2, SHARPNESS, 2, 1 / runs.received_every(manifest), still=STILL).to(DEVICE).eval()

## 1. Data

Every client tile at every received frame N from 2 on, of every clip (`runs.valid` frames only):
the 12 features, the wPSNR of repeat and of the search's N+1 over the region the client keeps,
whether the search's gate is open (moving) and whether N+1 differs from N (changed). A tenth of
the training sources - whole videos for the recordings - is held out to choose the training step.

In [2]:
CACHE.mkdir(parents=True, exist_ok=True)


def table(split):
    path = CACHE / f"{split}.pkl"
    if path.exists() and path.stat().st_mtime > (training.DATA_DIR / "manifest.json").stat().st_mtime:
        return pickle.loads(path.read_bytes())
    started = time.time()
    found = decide.load(manifest, split, published)
    path.write_bytes(pickle.dumps(found))
    print(f"{split}: {len(found['repeat'])} tiles in {time.time() - started:.0f}s")
    return found


def subset(found, mask):
    return {key: (value[mask] if isinstance(value, np.ndarray) and len(value) == len(mask) else value)
            for key, value in found.items()}


full = {split: table(split) for split in ("train", "val")}
for split, found in full.items():
    listed = manifest["splits"][split]["sources"]
    found["kind"] = np.array([listed[s]["kind"] for s in found["source"]])
    found["group"] = np.array([listed[s].get("video", listed[s]["file"]) for s in found["source"]])
groups = np.unique(full["train"]["group"])
held = set(np.random.default_rng(SEED).choice(groups, max(1, len(groups) // 10), replace=False))
in_held = np.isin(full["train"]["group"], list(held))
data = {"train": subset(full["train"], ~in_held), "held out": subset(full["train"], in_held), "val": full["val"]}
for name, found in data.items():
    m = found["moving"]
    better = np.minimum(found["search"], CAP) > np.minimum(found["repeat"], CAP)
    worse = np.minimum(found["repeat"], CAP) > np.minimum(found["search"], CAP)
    print(f"{name:<9} {len(m):6} tiles, {m.mean():4.0%} moving; of those the search better on {better[m].mean():4.0%}, "
          f"repeat on {worse[m].mean():4.0%} - " + ", ".join(f"{k} {(found['kind'] == k).sum()}" for k in KINDS))

train: 59856 tiles in 690s
val: 19374 tiles in 240s
train      54820 tiles,  61% moving; of those the search better on  57%, repeat on  14% - page 10556, desktop 9860, photo 6496, screen 17353, video 10555
held out    5036 tiles,  61% moving; of those the search better on  57%, repeat on  19% - page 1276, desktop 1276, photo 464, screen 1084, video 936
val        19374 tiles,  56% moving; of those the search better on  49%, repeat on  15% - page 3248, desktop 1392, photo 1740, screen 10652, video 2342


## 2. The ceiling and the threshold

On the moving tiles, capped at 50 dB: the search, repeat, the better of the two per tile, and the
search's gate raised - repeat where the cost at no motion is under a threshold, chosen on the
training tiles.

In [3]:
def chosen(found, take_search):
    """(S,) the capped wPSNR of picking the search's N+1 where `take_search`, repeat elsewhere; still tiles repeat."""
    return np.minimum(np.where(take_search & found["moving"], found["search"], found["repeat"]), CAP)


def zero_cost(found):
    return np.exp(found["features"][:, 0]) - 0.01


THRESHOLDS = [0.1, 0.25, 0.5, 0.75, 1, 1.5, 2, 3, 4, 6, 8]
print(f"{'moving tiles, ≤ 50 dB':<26}" + "".join(f"{name:>11}" for name in data))
for label, rule in (("repeat", lambda f: np.zeros(len(f["repeat"]), bool)),
                    ("search (gate 0.1)", lambda f: np.ones(len(f["repeat"]), bool)),
                    ("better of the two", lambda f: np.minimum(f["search"], CAP) >= np.minimum(f["repeat"], CAP))):
    print(f"{label:<26}" + "".join(f"{chosen(f, rule(f))[f['moving']].mean():11.2f}" for f in data.values()))
print()
for threshold in THRESHOLDS:
    print(f"{f'gate at {threshold} levels':<26}" + "".join(f"{chosen(f, zero_cost(f) >= threshold)[f['moving']].mean():11.2f}"
                                                          for f in data.values()))
GATE = max(THRESHOLDS, key=lambda th: chosen(data["train"], zero_cost(data["train"]) >= th)[data["train"]["moving"]].mean())
print(f"\nchosen on the training tiles: {GATE} levels")

moving tiles, ≤ 50 dB           train   held out        val
repeat                          36.30      34.64      37.62
search (gate 0.1)               37.91      35.34      38.80
better of the two               38.35      36.05      39.42

gate at 0.1 levels              37.91      35.34      38.80
gate at 0.25 levels             37.91      35.34      38.80
gate at 0.5 levels              37.90      35.34      38.80
gate at 0.75 levels             37.89      35.35      38.80
gate at 1 levels                37.88      35.35      38.78
gate at 1.5 levels              37.84      35.35      38.72
gate at 2 levels                37.81      35.35      38.68
gate at 3 levels                37.71      35.33      38.57
gate at 4 levels                37.64      35.33      38.51
gate at 6 levels                37.53      35.29      38.39
gate at 8 levels                37.41      35.25      38.34

chosen on the training tiles: 0.1 levels


## 3. The learned decision

Inputs: the 12 features (`gap` as log1p), standardised on the training tiles. Target: whether the
search's N+1 scores better than repeat on the tile, capped at 50 dB; weight: by how much. Moving
tiles only - a still tile repeats anyway. Logistic regression and the small network, Adam over the
whole set, kept at the step that scores best on the held-out tiles (moving, capped) - trained on
every kind, on the drawn kinds only and on the recorded kinds only.

In [4]:
def inputs(found):
    x = found["features"].copy()
    x[:, FEATURES.index("gap")] = np.log1p(x[:, FEATURES.index("gap")])
    return x


def fit(kinds, kind_of_net, steps_max=3000, rate=1e-2):
    train = data["train"]
    use = train["moving"] & np.isin(train["kind"], kinds)
    gain = np.minimum(train["search"], CAP) - np.minimum(train["repeat"], CAP)
    x, y, wgt = inputs(train)[use], (gain[use] > 0).astype(np.float32), np.abs(gain[use]).astype(np.float32)
    mean, std = x.mean(0), x.std(0) + 1e-6
    torch.manual_seed(SEED)
    net = torch.nn.Linear(len(FEATURES), 1) if kind_of_net == "logistic" else torch.nn.Sequential(
        torch.nn.Linear(len(FEATURES), 16), torch.nn.ReLU(), torch.nn.Linear(16, 16), torch.nn.ReLU(), torch.nn.Linear(16, 1))
    xt, yt, wt = (torch.as_tensor(a, dtype=torch.float32) for a in ((x - mean) / std, y, wgt / wgt.mean()))
    optimiser = torch.optim.Adam(net.parameters(), lr=rate)
    held_out = data["held out"]
    held_x = torch.as_tensor((inputs(held_out) - mean) / std, dtype=torch.float32)
    best = (-np.inf, None, 0)
    for step in range(steps_max + 1):
        if step % 25 == 0:
            with torch.no_grad():
                score = chosen(held_out, (net(held_x)[:, 0] > 0).numpy())[held_out["moving"]].mean()
            if score > best[0] + 1e-4:
                best = (score, {k: v.clone() for k, v in net.state_dict().items()}, step)
        loss = (torch.nn.functional.binary_cross_entropy_with_logits(net(xt)[:, 0], yt, reduction="none") * wt).mean()
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
    net.load_state_dict(best[1])

    def decide_tiles(found):
        with torch.no_grad():
            return (net(torch.as_tensor((inputs(found) - mean) / std, dtype=torch.float32))[:, 0] > 0).numpy()
    return decide_tiles, net, best


TRAINED = {}
for label, kinds in (("every kind", KINDS), ("drawn kinds", DRAWN), ("recorded kinds", RECORDED)):
    for kind_of_net in ("logistic", "network"):
        name = f"{kind_of_net}, {label}"
        TRAINED[name] = fit(kinds, kind_of_net)
        print(f"{name:<28} held out: {TRAINED[name][2][0]:.3f} dB (step {TRAINED[name][2][2]})")
held_out = data["held out"]
print(f"{'(the search)':<28} held out: {chosen(held_out, np.ones(len(held_out['repeat']), bool))[held_out['moving']].mean():.3f} dB")
weights = TRAINED["logistic, every kind"][1].weight.detach().numpy()[0]
print("\nlogistic regression on every kind, standardised weights (positive: carry the motion):")
for name, value in sorted(zip(FEATURES, weights), key=lambda p: -abs(p[1])):
    print(f"  {name:<20}{value:+7.2f}")

logistic, every kind         held out: 35.404 dB (step 800)
network, every kind          held out: 35.497 dB (step 125)
logistic, drawn kinds        held out: 35.400 dB (step 1225)
network, drawn kinds         held out: 35.475 dB (step 325)
logistic, recorded kinds     held out: 35.429 dB (step 625)
network, recorded kinds      held out: 35.332 dB (step 225)
(the search)                 held out: 35.343 dB

logistic regression on every kind, standardised weights (positive: carry the motion):
  explained             -0.91
  gap                   +0.76
  log best cost         +0.69
  residual share        -0.32
  change strength       -0.28
  peak                  -0.19
  log zero cost         -0.12
  unexplained cells     -0.07
  log motion            -0.06
  deblock               +0.05
  log bpp               +0.03
  changed pixels        -0.01


## 4. Validation by kind

The validation split - never trained on nor used to choose. Absolute wPSNR on the moving tiles
(capped) per kind and over all; the gain over the search on every moving tile with its 95% range
over videos (clips for the drawn kinds); and the share of moving tiles each rule sends to the search.

In [5]:
rng = np.random.default_rng(SEED)
found = data["val"]
moving, kind, group = found["moving"], found["kind"], found["group"]


def over(gain, mask, draws=2000):
    per = [gain[mask & (group == g)] for g in np.unique(group[mask])]
    means = [np.concatenate([per[i] for i in rng.integers(0, len(per), len(per))]).mean() for _ in range(draws)]
    return np.percentile(means, [2.5, 97.5])


RULES = {"repeat": lambda f: np.zeros(len(f["repeat"]), bool),
         "search (gate 0.1)": lambda f: np.ones(len(f["repeat"]), bool),
         f"gate raised to {GATE}": lambda f: zero_cost(f) >= GATE,
         **{f"learned: {name}": TRAINED[name][0] for name in TRAINED},
         "better of the two (sees N+1)": lambda f: np.minimum(f["search"], CAP) >= np.minimum(f["repeat"], CAP)}
base = chosen(found, np.ones(len(moving), bool))
print(f"{'moving tiles, ≤ 50 dB':<36}" + "".join(f"{k:>9}" for k in (*KINDS, "all")) + f"{'over the search':>17}{'95% over videos':>18}{'to search':>11}")
print(f"{'tiles':<36}" + "".join(f"{(moving & (kind == k)).sum():9}" for k in KINDS) + f"{moving.sum():9}")
for name, rule in RULES.items():
    take = rule(found) & moving
    score = chosen(found, take)
    gain = score - base
    low, high = over(gain, moving) if name != "search (gate 0.1)" else (0, 0)
    print(f"{name:<36}" + "".join(f"{score[moving & (kind == k)].mean():9.2f}" for k in KINDS) + f"{score[moving].mean():9.2f}"
          f"{gain[moving].mean():+12.2f}     [{low:+.2f}, {high:+.2f}]{take[moving].mean():10.0%}")

moving tiles, ≤ 50 dB                    page  desktop    photo   screen    video      all  over the search   95% over videos  to search
tiles                                    1810      332     1403     5115     2140    10800
repeat                                  32.06    38.88    31.12    42.74    34.13    37.62       -1.19     [-1.88, -0.49]        0%
search (gate 0.1)                       34.11    38.29    35.84    43.24    34.20    38.80       +0.00     [+0.00, +0.00]      100%
gate raised to 0.1                      34.11    38.29    35.84    43.24    34.20    38.80       +0.00     [+0.00, +0.00]      100%
learned: logistic, every kind           34.41    38.73    36.07    43.26    34.63    38.99       +0.19     [+0.04, +0.43]       58%
learned: network, every kind            34.57    38.74    36.14    43.25    34.64    39.03       +0.22     [+0.05, +0.48]       60%
learned: logistic, drawn kinds          34.32    38.72    35.84    43.22    34.32    38.86       +0.06     [-0.0

## What it says

- **A learned decision gains a third of the ceiling.** On every moving validation tile, choosing per
  tile between the search's N+1 and N is worth at most +0.61 dB over the search (seeing N+1). The
  small network trained on every kind takes **+0.22** of it (95% over videos +0.05 to +0.48) and
  logistic regression +0.19 (+0.04 to +0.43), sending about 60% of the moving tiles to the search.
- **It gains where a motion stops or is not a motion.** By kind, the network is +0.46 dB on pages,
  +0.45 on desktops, +0.44 on video, +0.30 on photos - and +0.01 on screens, where the search and
  repeat are already within 0.5 dB of each other (43.24 against 42.74) and the ceiling is +0.12.
- **Raising the search's gate does nothing.** Repeat wherever its cost at no motion is under a
  threshold, from 0.1 to 8 luma levels: the best is the published 0.1. The tiles where repeat wins
  are not quiet ones; something changed there that one motion gets wrong.
- **What it reads**: how much of the change the best motion explains (`explained`, by far), how
  clear the best match is against the next one (`gap`) and how good it is (`log best cost`), then
  the residual after aligning N−2. It asks the question the search already measures - does one
  motion account for this tile? - and answers "repeat" more readily where the match is ambiguous.
- **It needs every kind to learn from.** Trained on the drawn kinds alone it gains +0.14 (−0.09 to
  +0.41), on the recorded kinds alone the network loses −0.23 and the logistic one gains +0.15:
  each kind alone teaches a rule that does not carry to the other.

So a learned decision is worth keeping: twelve numbers the search mostly computes already and a
500-weight network, +0.2 dB on the moving tiles, no picture drawn that is not the search's or N.
It cannot see the stop that is not announced - the rest of the ceiling is what happens between N
and N+1.